# Part B2 — Binary transfer

**Model identity**

- Personal `p` = deployed offline ensemble `p_python_art_cfg` (0.6 / 0.4). Not LOSO predictions.
- Primary evaluation: chronological holdout (first 80% by timestamp = 64/17 when n=81).
- Random stratified repeats are secondary; SD is variation across splits, not a CI.
- Majority / prevalence baseline for Brier and log-loss uses the **training positive rate** as a constant probability. Hard majority accuracy is a separate column.
- When the logistic slope is negative, `slope_negative` is set and both `auc_raw` and `auc_calibrated` are stored. A high calibrated AUC with F1 = 0 is a reversed ranking, not a detection gain.

## Setup

In [ ]:
!pip -q install xgboost scikit-learn pandas numpy matplotlib seaborn

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/cloudnine-stress-ml')
    USER_CSV = PROJECT / 'data' / 'cloudNine-database-data.csv'
    P_CSV = PROJECT / 'data' / 'phone_vs_python_p.csv'
    LAB_CSV = PROJECT / 'data' / 'StressData.csv'
    OUTPUT_DIR = PROJECT / 'part_b_binary_results'
except Exception:
    PROJECT = Path.cwd()
    if not (PROJECT / 'docs').exists() and (PROJECT.parent / 'docs').exists():
        PROJECT = PROJECT.parent
    ART = PROJECT / 'docs' / 'Cloudnine Stress Artifacts'
    USER_CSV = ART / 'measurement_samples.csv'
    P_CSV = ART / 'phone_vs_python_p.csv'
    LAB_CSV = ART / 'StressData.csv'
    OUTPUT_DIR = ART / 'revision'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('USER', USER_CSV.exists(), '| LAB', LAB_CSV.exists(), '| OUT', OUTPUT_DIR)

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss, f1_score, log_loss,
    precision_score, recall_score, roc_auc_score,
)

FEATURES = ['hrrange', 'hrvar', 'hrstd', 'hrmin', 'hrmax', 'hrkurt']
LABEL_LAB = 'metric'
SUBJECT_LAB = 'subject'
RANDOM_SEED = 123
K_VALUES = [10, 20, 30, 40, 50]
MIN_TEST_SIZE = 17
N_REPEATS = 10
DECISION_THRESHOLD = 0.5
Y_BIN_CUTOFF = 5
USER_WEIGHT_RATIO = 1.0
LAB_MIX_MAX_ROWS = 40_000
MIX_XGB_ROUNDS = 200
HOLDOUT_TRAIN_FRAC = 0.8
MIX_XGB_PARAMS = {
    'eta': 0.5, 'max_depth': 8, 'subsample': 0.70, 'colsample_bytree': 0.8,
    'objective': 'binary:logistic', 'eval_metric': 'logloss', 'verbosity': 0,
}

np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', context='notebook')

## Load

In [ ]:
def load_user(csv_path, p_csv):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
    if 'is_test' in df.columns:
        df = df[~df['is_test'].astype(str).str.lower().isin(['true', '1', 'yes'])]
    if 'user_rating_status' in df.columns:
        df = df[df['user_rating_status'].astype(str).str.lower() == 'submitted']
    df['y10'] = pd.to_numeric(df['user_prediction'], errors='coerce')
    df = df[df['y10'].between(1, 10)].copy()
    if 'hr_sample_count' in df.columns:
        df = df[pd.to_numeric(df['hr_sample_count'], errors='coerce').fillna(0) >= 10]
    pdf = pd.read_csv(p_csv)
    pdf.columns = [c.strip().lower() for c in pdf.columns]
    df['id'] = df['id'].astype(str)
    pdf['id'] = pdf['id'].astype(str)
    df = df.merge(pdf[['id', 'p_python_art_cfg']], on='id', how='left')
    df['p'] = df['p_python_art_cfg'].astype(float)
    if df['p'].isna().any():
        raise RuntimeError('missing p')
    df = df.sort_values('timestamp').reset_index(drop=True)
    print('p source: p_python_art_cfg (deployed 0.6/0.4)')
    return df


def load_lab(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}
    out = pd.DataFrame({f: df[lower.get(f, f)].astype(float) for f in FEATURES})
    out[LABEL_LAB] = df[lower['metric']].astype(int)
    out[SUBJECT_LAB] = df[lower.get('subject', 'Subject')].astype(str)
    return out


user_df = load_user(USER_CSV, P_CSV)
lab_df = load_lab(LAB_CSV)
assert len(user_df) == 81
CHRONO_SPLIT = int(len(user_df) * HOLDOUT_TRAIN_FRAC)
assert CHRONO_SPLIT == 64
print('user', len(user_df), '| chrono split', CHRONO_SPLIT, '/', len(user_df) - CHRONO_SPLIT, '| lab', len(lab_df))

## Helpers

In [ ]:
def classification_metrics(y_true, p, threshold=DECISION_THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    p = np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7)
    y_hat = (p >= threshold).astype(int)
    out = {
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'precision': float(precision_score(y_true, y_hat, zero_division=0)),
        'recall': float(recall_score(y_true, y_hat, zero_division=0)),
        'f1': float(f1_score(y_true, y_hat, zero_division=0)),
        'auc': np.nan,
        'brier': float(brier_score_loss(y_true, p)),
        'log_loss': float(log_loss(y_true, p, labels=[0, 1])),
        'n_test': int(len(y_true)),
        'n_pos_test': int(y_true.sum()),
    }
    if len(np.unique(y_true)) >= 2:
        out['auc'] = float(roc_auc_score(y_true, p))
    return out


def fit_logistic(p_train, y_train):
    p_train = np.asarray(p_train, float).reshape(-1, 1)
    y_train = np.asarray(y_train).astype(int)
    if len(np.unique(y_train)) < 2:
        prior = float(np.clip(y_train.mean(), 1e-3, 1 - 1e-3))
        return ('constant', prior, None)
    clf = LogisticRegression(solver='lbfgs', max_iter=1000)
    clf.fit(p_train, y_train)
    return ('logistic', clf, float(clf.coef_[0, 0]))


def apply_cal(cal, p):
    kind, obj, _ = cal
    p = np.asarray(p, float)
    if kind == 'constant':
        return np.full(len(p), obj, float)
    return obj.predict_proba(p.reshape(-1, 1))[:, 1]


def subsample_lab(lab, rng, max_rows):
    if max_rows is None or len(lab) <= max_rows:
        return lab
    y = lab[LABEL_LAB].astype(int).values
    idx0, idx1 = np.where(y == 0)[0], np.where(y == 1)[0]
    frac = max_rows / len(lab)
    n1 = max(1, int(round(len(idx1) * frac)))
    n0 = min(max(max_rows - n1, 1), len(idx0))
    n1 = min(max_rows - n0, len(idx1))
    take = np.concatenate([rng.choice(idx0, n0, False), rng.choice(idx1, n1, False)])
    return lab.iloc[take]


def scale_pos_weight(y):
    y = np.asarray(y)
    return int((y == 0).sum()) / max(int((y == 1).sum()), 1)


def train_xgb(X, y, w, rng):
    params = MIX_XGB_PARAMS.copy()
    params['seed'] = int(rng.integers(0, 10_000_000))
    params['scale_pos_weight'] = scale_pos_weight(y)
    return xgb.train(
        params,
        xgb.DMatrix(X, label=y, weight=w, feature_names=FEATURES),
        num_boost_round=MIX_XGB_ROUNDS, verbose_eval=False,
    )


def predict_xgb(model, X):
    return np.clip(model.predict(xgb.DMatrix(np.asarray(X, float), feature_names=FEATURES)), 1e-7, 1 - 1e-7)


def train_lab_xgb(lab_part, rng):
    X = lab_part[FEATURES].astype(float).values
    y = lab_part[LABEL_LAB].astype(int).values
    return train_xgb(X, y, np.ones(len(y)), rng)


def train_mix_xgb(lab_part, X_user, y_user, rng):
    X_lab = lab_part[FEATURES].astype(float).values
    y_lab = lab_part[LABEL_LAB].astype(int).values
    n_lab, k = len(y_lab), len(y_user)
    w = np.concatenate([np.ones(n_lab), np.full(k, USER_WEIGHT_RATIO * n_lab / k)])
    return train_xgb(np.vstack([X_lab, X_user]), np.concatenate([y_lab, y_user]), w, rng)


def oof_mix_scores(lab, X_user, y_user, rng, n_folds=5):
    k = len(y_user)
    oof = np.full(k, np.nan)
    for fold_idx in np.array_split(np.arange(k), n_folds):
        train_idx = np.setdiff1d(np.arange(k), fold_idx)
        lab_part = subsample_lab(lab, rng, LAB_MIX_MAX_ROWS)
        if len(train_idx) < 2 or len(np.unique(y_user[train_idx])) < 2:
            model = train_lab_xgb(lab_part, rng)
        else:
            model = train_mix_xgb(lab_part, X_user[train_idx], y_user[train_idx], rng)
        oof[fold_idx] = predict_xgb(model, X_user[fold_idx])
    return oof


def stratified_k_split(y, k, rng):
    y = np.asarray(y).astype(int)
    idx0 = np.where(y == 0)[0].copy(); idx1 = np.where(y == 1)[0].copy()
    rng.shuffle(idx0); rng.shuffle(idx1)
    n1 = int(np.clip(np.round(k * y.mean()), 1, k - 1))
    n1 = min(max(n1, 1), len(idx1) - 1)
    n0 = min(max(k - n1, 1), len(idx0) - 1)
    n1 = k - n0
    train_idx = np.concatenate([idx0[:n0], idx1[:n1]])
    rng.shuffle(train_idx)
    mask = np.ones(len(y), bool); mask[train_idx] = False
    return train_idx, np.where(mask)[0]

## Primary: chronological holdout

In [ ]:
X_all = user_df[FEATURES].astype(float).values
p_all = user_df['p'].astype(float).values
y10 = user_df['y10'].values

chrono_rows = []
for cut, cut_name in [(4, '3_4'), (5, '4_5'), (6, '5_6')]:
    y_bin = (y10 >= cut).astype(int)
    tr = np.arange(CHRONO_SPLIT)
    te = np.arange(CHRONO_SPLIT, len(y_bin))
    y_tr, y_te = y_bin[tr], y_bin[te]
    p_tr, p_te = p_all[tr], p_all[te]
    X_tr, X_te = X_all[tr], X_all[te]
    rng = np.random.default_rng(RANDOM_SEED + cut * 17)

    train_prev = float(y_tr.mean())  # prevalence baseline probability
    hard_maj = int(train_prev >= 0.5)
    maj_acc = float((y_te == hard_maj).mean())
    p_prev = np.full(len(y_te), np.clip(train_prev, 1e-7, 1 - 1e-7))
    m_prev = classification_metrics(y_te, p_prev)
    chrono_rows.append({
        'method': 'Prevalence_train', 'protocol': 'chronological_holdout',
        'threshold_cut': cut_name, 'n_train': len(y_tr),
        'train_prevalence': train_prev, 'majority_acc': maj_acc,
        'slope_sign': None, 'slope_negative': False,
        'auc_raw': np.nan, 'auc_calibrated': m_prev['auc'],
        **m_prev,
    })

    m0 = classification_metrics(y_te, p_te)
    chrono_rows.append({
        'method': 'M0', 'protocol': 'chronological_holdout',
        'threshold_cut': cut_name, 'n_train': len(y_tr),
        'train_prevalence': train_prev, 'majority_acc': maj_acc,
        'slope_sign': None, 'slope_negative': False,
        'auc_raw': m0['auc'], 'auc_calibrated': m0['auc'],
        **m0,
    })

    cal = fit_logistic(p_tr, y_tr)
    p_m1 = apply_cal(cal, p_te)
    m1 = classification_metrics(y_te, p_m1)
    slope = cal[2]
    slope_neg = bool(slope is not None and slope < 0)
    if slope is not None and slope > 0:
        assert abs(m1['auc'] - m0['auc']) < 1e-12, (cut_name, m1['auc'], m0['auc'])
        print(f'ASSERT M1 AUC == M0 AUC [{cut_name}] OK ({m0["auc"]:.6f})')
    chrono_rows.append({
        'method': 'M1', 'protocol': 'chronological_holdout',
        'threshold_cut': cut_name, 'n_train': len(y_tr),
        'train_prevalence': train_prev, 'majority_acc': maj_acc,
        'slope_sign': slope, 'slope_negative': slope_neg,
        'auc_raw': m0['auc'], 'auc_calibrated': m1['auc'],
        'cal_kind': cal[0], **m1,
    })

    lab_part = subsample_lab(lab_df, rng, LAB_MIX_MAX_ROWS)
    p_lab = predict_xgb(train_lab_xgb(lab_part, rng), X_te)
    m_lab = classification_metrics(y_te, p_lab)
    chrono_rows.append({
        'method': 'LabXGB', 'protocol': 'chronological_holdout',
        'threshold_cut': cut_name, 'n_train': len(y_tr),
        'train_prevalence': train_prev, 'majority_acc': maj_acc,
        'slope_sign': None, 'slope_negative': False,
        'auc_raw': m_lab['auc'], 'auc_calibrated': m_lab['auc'],
        **m_lab,
    })

    mix = train_mix_xgb(lab_part, X_tr, y_tr, rng)
    p_mix = predict_xgb(mix, X_te)
    m_mix = classification_metrics(y_te, p_mix)
    chrono_rows.append({
        'method': 'Mix', 'protocol': 'chronological_holdout',
        'threshold_cut': cut_name, 'n_train': len(y_tr),
        'train_prevalence': train_prev, 'majority_acc': maj_acc,
        'slope_sign': None, 'slope_negative': False,
        'auc_raw': m_mix['auc'], 'auc_calibrated': m_mix['auc'],
        **m_mix,
    })

    if cut == 5:
        oof = oof_mix_scores(lab_df, X_tr, y_tr, rng)
        cal_mix = fit_logistic(oof, y_tr)
        p_mix_cal = apply_cal(cal_mix, p_mix)
        m_mc = classification_metrics(y_te, p_mix_cal)
        slope_m = cal_mix[2]
        slope_neg_m = bool(slope_m is not None and slope_m < 0)
        chrono_rows.append({
            'method': 'Mix+cal_OOF', 'protocol': 'chronological_holdout',
            'threshold_cut': cut_name, 'n_train': len(y_tr),
            'train_prevalence': train_prev, 'majority_acc': maj_acc,
            'slope_sign': slope_m, 'slope_negative': slope_neg_m,
            'auc_raw': m_mix['auc'], 'auc_calibrated': m_mc['auc'],
            'cal_kind': cal_mix[0], **m_mc,
        })

chrono = pd.DataFrame(chrono_rows)
chrono.to_csv(OUTPUT_DIR / 'b2_chronological_holdout.csv', index=False)

# Identity checks on primary 4/5
c45 = chrono[chrono.threshold_cut == '4_5']
prev = c45[c45.method == 'Prevalence_train'].iloc[0]
m1 = c45[c45.method == 'M1'].iloc[0]
m0 = c45[c45.method == 'M0'].iloc[0]
mixc = c45[c45.method == 'Mix+cal_OOF'].iloc[0]
print('4/5 Prevalence Brier/logloss', round(prev.brier, 3), round(prev.log_loss, 3),
      'train_prev', round(prev.train_prevalence, 3))
print('4/5 M1 Brier/logloss', round(m1.brier, 3), round(m1.log_loss, 3),
      '(compare to prevalence, not 0.01)')
print('M1 vs prevalence Brier: M1', m1.brier, 'prev', prev.brier,
      'M1_worse_than_prevalence', m1.brier > prev.brier)
# Do not assert M1_worse: that is an empirical finding on this holdout, not an invariant.
# Keep the positive-slope check above: M1 AUC must equal M0 AUC when slope > 0.
print('Mix+cal slope_negative', bool(mixc.slope_negative), 'slope', mixc.slope_sign,
      'auc_raw', mixc.auc_raw, 'auc_calibrated', mixc.auc_calibrated)
if bool(mixc.slope_negative):
    print('NOTE: Mix+cal AUC is a reversed ranking of Mix, not a detection gain; F1 stays 0')
display(c45[[
    'method', 'f1', 'accuracy', 'auc', 'auc_raw', 'auc_calibrated',
    'brier', 'log_loss', 'majority_acc', 'train_prevalence',
    'slope_sign', 'slope_negative',
]])

## Secondary: random stratified repeats

In [ ]:
# SD = variation across overlapping random splits, not a confidence interval.
y_bin = (user_df['y10'].values >= Y_BIN_CUTOFF).astype(int)
n_user = len(user_df)
k_grid = sorted(set([k for k in K_VALUES if k < n_user - MIN_TEST_SIZE] + [n_user - MIN_TEST_SIZE]))
rows = []
for k in k_grid:
    for rep in range(N_REPEATS):
        rng = np.random.default_rng(RANDOM_SEED + 1000 * k + rep)
        tr, te = stratified_k_split(y_bin, k, rng)
        y_tr, y_te = y_bin[tr], y_bin[te]
        p_tr, p_te = p_all[tr], p_all[te]
        X_tr, X_te = X_all[tr], X_all[te]

        rows.append({'method': 'M0', 'k': k, 'repeat': rep, **classification_metrics(y_te, p_te)})
        cal = fit_logistic(p_tr, y_tr)
        p_m1 = apply_cal(cal, p_te)
        m1 = classification_metrics(y_te, p_m1)
        rows.append({
            'method': 'M1', 'k': k, 'repeat': rep,
            'slope_sign': cal[2],
            'slope_negative': bool(cal[2] is not None and cal[2] < 0),
            **m1,
        })
        lab_part = subsample_lab(lab_df, rng, LAB_MIX_MAX_ROWS)
        p_mix = predict_xgb(train_mix_xgb(lab_part, X_tr, y_tr, rng), X_te)
        rows.append({'method': 'Mix', 'k': k, 'repeat': rep, **classification_metrics(y_te, p_mix)})
        print(f'k={k} rep={rep} done')

results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / 'b2_random_secondary_raw.csv', index=False)

summary_rows = []
for (method, k), g in results.groupby(['method', 'k']):
    row = {
        'method': method, 'k': k, 'n_repeats': len(g),
        'note': 'random_stratified_secondary; sd=variation_across_splits_not_CI',
    }
    for m in ['f1', 'auc', 'brier', 'log_loss', 'accuracy']:
        row[f'{m}_mean'] = g[m].mean()
        row[f'{m}_std'] = g[m].std(ddof=1)
    if 'slope_negative' in g.columns:
        row['frac_negative_slope'] = float(g['slope_negative'].fillna(False).mean())
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / 'b2_random_secondary_summary.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for method in ['M0', 'M1', 'Mix']:
    sub = summary[summary.method == method].sort_values('k')
    axes[0].plot(sub.k, sub.f1_mean, marker='o', markersize=3.5, label=method, lw=1.4)
    axes[1].plot(sub.k, sub.auc_mean, marker='o', markersize=3.5, label=method, lw=1.4)
axes[0].set_xlabel('$k$'); axes[0].set_ylabel('F1 mean')
axes[1].set_xlabel('$k$'); axes[1].set_ylabel('AUC mean')
axes[0].set_title('Secondary: F1 vs $k$ (SD = split variation)')
axes[1].set_title('Secondary: AUC vs $k$ (SD = split variation)')
for ax in axes:
    ax.grid(True, alpha=0.3); ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig_f1_auc_vs_k.png', dpi=150)
plt.show()
display(summary[['method', 'k', 'f1_mean', 'f1_std', 'auc_mean', 'auc_std']].round(4))
print('Primary table: b2_chronological_holdout.csv')